# Analyse des Artikel-Klassifikators (BiLSTM)

Dieses Notebook evaluiert den trainierten **Artikel-Klassifikator** (AS vs. LS) auf Dokumentenebene:
1. Laden des Modells und der Gewichte.
2. Rekonstruktion/Laden des Vokabulars.
3. Evaluierung auf Test- und Validierungsdaten.
4. Interaktive Klassifikation eigener Texte.


In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import spacy
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

while not os.path.exists(".git"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir("..")
print("Arbeitsverzeichnis:", os.getcwd())


## 1. Konfiguration & Hyperparameter
Bitte passe die Werte so an, wie sie beim Training von `2_binary_train_article_model.py` verwendet wurden.


In [ ]:
CSV_PATH = "data/analysis/corpus_master.csv"
MODEL_PATH = "results/models/best_model_sim_0.8_0.98.pt"  # Achtung: Pfad anpassen! Z.B. best_model_sim_... oder lstm_article_sim_...
MIN_SIM = 0.8
MAX_SIM = 0.98
MAX_SEQ_LEN = 512
EMBEDDING_DIM = 128
HIDDEN_DIM = 128

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("Nutze Device:", DEVICE)


## 2. Daten laden & Vokabular rekonstruieren


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

class Vocab:
    def __init__(self, sentences, max_size=25000, min_freq=3):
        counter = Counter()
        for sent in sentences:
            counter.update(sent)
        self.itos = ["<pad>", "<unk>"]
        self.stoi = {"<pad>": 0, "<unk>": 1}
        for token, freq in counter.most_common(max_size):
            if freq >= min_freq:
                self.stoi[token] = len(self.itos)
                self.itos.append(token)
    def __len__(self): return len(self.itos)
    def encode(self, tokens):
        return [self.stoi.get(t, self.stoi["<unk>"]) for t in tokens]

print("Lade Artikel (dauert kurz)...")
df = pd.read_csv(CSV_PATH)
mask = (df["semantic_similarity_8192"] >= MIN_SIM) & (df["semantic_similarity_8192"] <= MAX_SIM)
df_filtered = df[mask]

nlp = spacy.load("de_core_news_sm", disable=["ner", "tagger", "lemmatizer"])

ls_articles = []
as_articles = []
for _, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    ls_tokens = [t.text.lower() for t in nlp(str(row["ls_text"])) if not t.is_space]
    as_tokens = [t.text.lower() for t in nlp(str(row["as_text"])) if not t.is_space]
    if ls_tokens: ls_articles.append(ls_tokens)
    if as_tokens: as_articles.append(as_tokens)

X = ls_articles + as_articles
y = [1] * len(ls_articles) + [0] * len(as_articles)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.11, random_state=42, stratify=y_train_val)

vocab = Vocab(X_train)
print(f"Rekonstruierte Vokabular-Größe: {len(vocab)}")
print(f"Test-Artikel: {len(X_test)}")


## 3. Modell definieren & laden


In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim=1):
        super(BiLSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        return self.fc(self.dropout(hidden))

model = BiLSTMClassifier(len(vocab), EMBEDDING_DIM, HIDDEN_DIM).to(DEVICE)
# Versuche verschiedene mögliche Speicherpfade zu prüfen
import glob
matching_models = glob.glob("results/models/lstm_article_sim_*.pt") + [MODEL_PATH]
loaded = False
for path in matching_models:
    if os.path.exists(path):
        try:
            model.load_state_dict(torch.load(path, map_location=DEVICE))
            print(f"Erfolgreich Modell-Gewichte von {path} geladen!")
            loaded = True
            break
        except Exception as e:
            continue

if not loaded:
    print("WARNUNG: Kein passendes Modell geladen. Bitte überprüfen Sie den MODEL_PATH!")
model.eval()


## 4. Evaluierung auf den Testdaten


In [ ]:
all_preds = []
all_targets = []

with torch.no_grad():
    for x_tokens, label in zip(X_test, y_test):
        encoded = vocab.encode(x_tokens)[:MAX_SEQ_LEN]
        padded = encoded + [0] * (MAX_SEQ_LEN - len(encoded))
        inp = torch.tensor([padded], dtype=torch.long).to(DEVICE)
        
        logits = model(inp)
        prob = torch.sigmoid(logits).item()
        pred = 1 if prob >= 0.5 else 0
        
        all_preds.append(pred)
        all_targets.append(label)

print("--- Klassifikationsbericht ---")
print(classification_report(all_targets, all_preds, target_names=["AS (Alltagssprache)", "LS (Leichte Sprache)"]))
print("Accuracy:", accuracy_score(all_targets, all_preds))
print("Balanced Accuracy:", balanced_accuracy_score(all_targets, all_preds))


## 5. Eigene Texte testen


In [ ]:
def predict_article(text):
    tokens = [t.text.lower() for t in nlp(text) if not t.is_space]
    encoded = vocab.encode(tokens)[:MAX_SEQ_LEN]
    padded = encoded + [0] * (MAX_SEQ_LEN - len(encoded))
    inp = torch.tensor([padded], dtype=torch.long).to(DEVICE)
    
    with torch.no_grad():
        logits = model(inp)
        prob = torch.sigmoid(logits).item()
    
    label = "Leichte Sprache (LS)" if prob >= 0.5 else "Alltagssprache (AS)"
    print(f"Text: {text[:200]}...")
    print(f"-> Wahrscheinlichkeit für Leichte Sprache: {prob:.4f}")
    print(f"-> Klasse: {label}\n")

predict_article("Das Bundesministerium für Gesundheit stellt neue Richtlinien vor. Bürger können sich ab sofort online informieren.")
predict_article("Hier finden Sie Informationen in Leichter Sprache. Wir erklären Ihnen den Weg zum Amt. Das Amt hilft den Bürgern.")
